# Labor Demand Estimation

Estimates the labor demand equation at the **firm × occupation × year** level:

$$
l_i = \underbrace{\left(\frac{\tau_a}{1+\tau\tau_e}\right)\bar{a}}_{\alpha} + w_{-i} + \tau a_i
$$

where:
- $l_i$ = log job-posting count for firm $i$ in occupation $o$, year $t$
- $w_{-i}$ = **leave-one-out mean wage**: average posted wage in occupation $o$, year $t$, excluding firm $i$
- $\tau a_i$ = residual whose distribution is the object of interest

Four specifications are estimated (2 wage measures × 2 wage samples):

| | All wages (incl. Revelio imputations) | Observed wages only |
|---|---|---|
| $w_{-i}$ in levels | (1) | (3) |
| $\log w_{-i}$ | (2) | (4) |

## 0. Parameters

In [ ]:
# Occupation granularity: 'role_k10' | 'role_k50' | 'role_k150' | 'onet_code'
OCC_VAR = 'role_k50'

# Minimum number of OTHER firms in occ-year cell for LOO wage to be meaningful
MIN_OTHER_FIRMS = 2

## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/wage-as-signal')
except ImportError:
    ROOT = Path.cwd()
    if ROOT.name in ('paper', 'code'):
        ROOT = ROOT.parent

OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)

## 2. Load data

In [ ]:
raw = pd.read_csv(
    ROOT / 'data/raw/revelio_academic_postings/academic_postings_unified_individual_academic.csv',
    low_memory=False
)
raw['year'] = pd.to_datetime(raw['post_date'], errors='coerce').dt.year

# salary_predicted is stored as string 'True'/'False' in the unified file
raw['salary_observed'] = raw['salary_predicted'].astype(str).str.strip().str.lower() == 'false'

print(f"Total postings : {len(raw):,}")
print(f"Observed wages : {raw['salary_observed'].sum():,}  ({100*raw['salary_observed'].mean():.1f}%)")
print(f"Imputed wages  : {(~raw['salary_observed']).sum():,}  ({100*(~raw['salary_observed']).mean():.1f}%)")

## 3. Build firm × occupation × year panel

In [ ]:
def build_panel(df):
    """Aggregate to firm-occ-year level and compute leave-one-out wage."""
    d = df.dropna(subset=['rcid', OCC_VAR, 'year', 'salary']).copy()
    d = d[d['salary'] > 0]
    d['rcid'] = d['rcid'].astype(int)
    d['year'] = d['year'].astype(int)

    firm_panel = (
        d.groupby(['rcid', OCC_VAR, 'year'])
        .agg(posting_count=('job_id', 'count'),
             firm_wage_sum=('salary', 'sum'),
             firm_n_wages=('salary', 'count'))
        .reset_index()
    )
    occ_totals = (
        d.groupby([OCC_VAR, 'year'])
        .agg(occ_wage_sum=('salary', 'sum'),
             occ_n_wages=('salary', 'count'),
             occ_n_firms=('rcid', 'nunique'))
        .reset_index()
    )
    p = firm_panel.merge(occ_totals, on=[OCC_VAR, 'year'])

    # Leave-one-out wage: occupation mean excluding the current firm's postings
    loo_n           = p['occ_n_wages'] - p['firm_n_wages']
    p['w_loo']      = (p['occ_wage_sum'] - p['firm_wage_sum']) / loo_n
    p['loo_n']      = loo_n
    p['log_w_loo']  = np.log(p['w_loo'])
    p['log_count']  = np.log(p['posting_count'])

    p = p[(p['occ_n_firms'] - 1) >= MIN_OTHER_FIRMS]
    p = p[p['loo_n'] > 0].dropna(subset=['w_loo', 'log_w_loo'])
    return p


all_panel = build_panel(raw)
obs_panel = build_panel(raw[raw['salary_observed']])

print(f"All-wages panel  : {len(all_panel):,} cells, {all_panel['rcid'].nunique()} unique firms")
print(f"Observed panel   : {len(obs_panel):,} cells, {obs_panel['rcid'].nunique()} unique firms")

## 4. OLS estimation — four specifications

In [ ]:
def run_ols(panel, w_col, label):
    y = panel['log_count']
    X = sm.add_constant(panel[w_col])
    m = sm.OLS(y, X).fit(cov_type='HC3')
    m._label = label
    return m

m1 = run_ols(all_panel, 'w_loo',     '(1) Levels / All')
m2 = run_ols(all_panel, 'log_w_loo', '(2) Log   / All')
m3 = run_ols(obs_panel, 'w_loo',     '(3) Levels / Observed')
m4 = run_ols(obs_panel, 'log_w_loo', '(4) Log   / Observed')
models = [m1, m2, m3, m4]

In [ ]:
# ── Comparison table ────────────────────────────────────────────────────────
def stars(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''

headers = ['(1) Levels\nAll wages', '(2) Log\nAll wages',
           '(3) Levels\nObserved', '(4) Log\nObserved']
wvars   = ['w_loo', 'log_w_loo', 'w_loo', 'log_w_loo']

rows = {}
for i, (m, wv, h) in enumerate(zip(models, wvars, headers)):
    rows[h] = {
        'Constant'           : f"{m.params['const']:.4f}{stars(m.pvalues['const'])}",
        'Constant SE'        : f"({m.bse['const']:.4f})",
        'w_loo coef'         : f"{m.params[wv]:.2e}{stars(m.pvalues[wv])}",
        'w_loo SE'           : f"({m.bse[wv]:.2e})",
        'R²'                 : f"{m.rsquared:.4f}",
        'N'                  : f"{int(m.nobs):,}",
    }

display(pd.DataFrame(rows))
print('HC3 robust SEs in parentheses. * p<0.10  ** p<0.05  *** p<0.01')

## 5. Residual distribution $\tau a_i$

In [ ]:
# ── Summary statistics across specs ────────────────────────────────────────
resid_summary = {}
for m, h in zip(models, headers):
    r = m.resid
    jb_stat, jb_p = stats.jarque_bera(r)
    resid_summary[h.replace('\n', ' ')] = {
        'Std dev'        : r.std(),
        'Skewness'       : r.skew(),
        'Ex. kurtosis'   : r.kurtosis(),
        'p10'            : r.quantile(0.10),
        'p25'            : r.quantile(0.25),
        'Median'         : r.median(),
        'p75'            : r.quantile(0.75),
        'p90'            : r.quantile(0.90),
        'JB p-value'     : jb_p,
    }

display(pd.DataFrame(resid_summary).round(4))

In [ ]:
# ── Residual plots (2 × 2 grid, one panel per spec) ────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle(r'Residual distribution $\tau a_i$ — four specifications', fontsize=13)

for col, (m, h) in enumerate(zip(models, headers)):
    resid  = m.resid
    x_grid = np.linspace(resid.min(), resid.max(), 300)
    mu, sigma = stats.norm.fit(resid)

    # Top row: histogram + normal overlay
    ax = axes[0, col]
    ax.hist(resid, bins=35, density=True, color='steelblue', alpha=0.65)
    ax.plot(x_grid, stats.norm.pdf(x_grid, mu, sigma), 'r--', lw=1.5)
    ax.set_title(h.replace('\n', ' — '), fontsize=9)
    ax.set_xlabel(r'$\hat{\epsilon}_i$', fontsize=8)
    if col == 0: ax.set_ylabel('Density', fontsize=8)

    # Bottom row: Q-Q plot
    ax = axes[1, col]
    stats.probplot(resid, plot=ax)
    ax.set_title('')
    ax.get_lines()[1].set(color='red', lw=1.5)
    if col == 0: ax.set_ylabel('Sample quantiles', fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUTS / 'fig_residual_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved outputs/fig_residual_distribution.png')

## 6. Save outputs for paper

In [ ]:
# ── Regression table as .tex ───────────────────────────────────────────────
# Build manually so all 4 columns appear in one table
def make_tex_table(models, wvars, col_labels):
    def s(p):
        if p<0.01: return '$^{***}$'
        if p<0.05: return '$^{**}$'
        if p<0.10: return '$^{*}$'
        return ''

    lines = [
        r'\begin{table}[htbp]',
        r'\centering',
        r'\caption{Labor demand estimates: $l_i = \alpha + \hat{\beta}\,w_{-i} + \hat{\epsilon}_i$}',
        r'\label{tab:labor_demand}',
        r'\begin{tabular}{lcccc}',
        r'\hline\hline',
        r' & (1) & (2) & (3) & (4) \\',
        r' & Levels & Log $w$ & Levels & Log $w$ \\',
        r' & All wages & All wages & Observed & Observed \\',
        r'\hline',
    ]
    for lbl, param in [('Constant', 'const'), ('$w_{-i}$', None)]:
        coef_row = f'{lbl}'
        se_row   = ''
        for m, wv in zip(models, wvars):
            pv = param if param else wv
            c  = m.params[pv]
            se = m.bse[pv]
            p  = m.pvalues[pv]
            coef_row += f' & ${c:.4f}${s(p)}' if abs(c)>0.0001 else f' & ${c:.2e}${s(p)}'
            se_row   += f' & $({se:.4f})$' if abs(se)>0.0001 else f' & $({se:.2e})$'
        lines.append(coef_row + r' \\')
        lines.append(se_row   + r' \\')
    lines += [
        r'\hline',
        r'$R^2$ & ' + ' & '.join(f'{m.rsquared:.4f}' for m in models) + r' \\',
        r'$N$ & ' + ' & '.join(f'{int(m.nobs):,}' for m in models) + r' \\',
        r'\hline\hline',
        r'\multicolumn{5}{l}{\footnotesize HC3 robust standard errors in parentheses.'
        r' $^{*}p<0.10$, $^{**}p<0.05$, $^{***}p<0.01$.}\\',
        r'\end{tabular}',
        r'\end{table}',
    ]
    return '\n'.join(lines)

tex = make_tex_table(models, wvars, headers)
(OUTPUTS / 'table_labor_demand.tex').write_text(tex, encoding='utf-8')
print('Saved outputs/table_labor_demand.tex')

# ── Residual stats table as .tex ───────────────────────────────────────────
resid_df = pd.DataFrame(resid_summary).round(4)
resid_df.columns = ['(1)', '(2)', '(3)', '(4)']
tex_r = (
    r'\begin{table}[htbp]' + '\n'
    r'\centering' + '\n'
    r'\caption{Distribution of residual $\tau a_i$}' + '\n'
    r'\label{tab:residual_dist}' + '\n'
    + resid_df.to_latex(escape=False) +
    r'\end{table}'
)
(OUTPUTS / 'table_residual_dist.tex').write_text(tex_r, encoding='utf-8')
print('Saved outputs/table_residual_dist.tex')

## 7. Use in paper

```latex
\input{outputs/table_labor_demand}
\input{outputs/table_residual_dist}
\includegraphics[width=\textwidth]{outputs/fig_residual_distribution}
```

To export this notebook as HTML for email:
```
jupyter nbconvert --to html --no-input code/01_labor_demand.ipynb --output paper/labor_demand.html
```